# <font face="Verdana" size=6 color='#6495ED'> IAD-004 APRENDIZAGEM DE MÁQUINAS 1

 <font face="Verdana" size=3 color='#40E0D0'> Professores Larissa Driemeier e Thiago de Castro Martins

<center><img src='https://drive.google.com/uc?export=view&id=1sRJW7uRq_SqpfXEcMNwdaA9MQCj1V1qx' width="600"></center>

Este Notebook de autoestudo é uma recordação de *Decomposição em valor singular*, aula da disciplina IAD-001.

# Importando bibliotecas

In [ ]:
%matplotlib inline
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd
from google.colab import files
import numpy as np
from sklearn.decomposition import TruncatedSVD
import math



In [ ]:
color_g1 = 'darkolivegreen'
color_g2 = 'olive'
color_g3 = 'forestgreen'
color_g4 = 'green'
color_p1 = 'darkmagenta'
color_p2 = 'mediumorchid'
color_p3 = 'thistle'
color_p4 = 'lavandablush'
color_b1 = 'navy'
color_b2 = 'royalblue'
color_b4 = 'slateblue'
color_r1 = 'indianred'

# SVD

Esta parte do Notebook tem como objetivo recordar a Decomposição em Valores Singulares (SVD), um método amplamente utilizado para fatorar matrizes. Dada uma matriz qualquer $\boldsymbol{A}_{m \times n}$, sua decomposição é dada por

$$
\boldsymbol{A}=\boldsymbol{U}\boldsymbol{\Sigma}\boldsymbol{V}^T.
$$

Os vetores singulares à esquerda, denotados por $\boldsymbol{u}_i$, são os autovetores da matriz

$$
\boldsymbol{A}\boldsymbol{A}^T,
$$

e formam as colunas da matriz $\boldsymbol{U}$.

De forma análoga, os vetores singulares à direita, denotados por $\boldsymbol{v}_i$, são os autovetores da matriz
$$
\boldsymbol{A}^T\boldsymbol{A},
$$

e formam as colunas da matriz $\boldsymbol{V}$.

Como $\boldsymbol{A}\boldsymbol{A}^T$ e $\boldsymbol{A}^T\boldsymbol{A}$ são matrizes simétricas, seus autovetores são ortogonais e podem ser transformados em versores (norma unitária). Consequentemente, $\boldsymbol{U}$ e $\boldsymbol{V}$ são matrizes ortogonais, satisfazendo
$$
\boldsymbol{U}^T\boldsymbol{U}=\boldsymbol{I}
$$
e
$$
\boldsymbol{V}^T\boldsymbol{V}=\boldsymbol{I}.
$$

Os elementos da matriz diagonal $\boldsymbol{\Sigma}$ são chamados de valores singulares. Se $\lambda_i$ representa um autovalor não nulo de $\boldsymbol{A}\boldsymbol{A}^T$ (ou, equivalentemente, de $\boldsymbol{A}^T\boldsymbol{A}$), então o valor singular correspondente é dado por
$$
\sigma_i=\sqrt{\lambda_i}.
$$

É importante destacar que as matrizes $\boldsymbol{A}\boldsymbol{A}^T$ e $\boldsymbol{A}^T\boldsymbol{A}$ possuem os mesmos autovalores não nulos e, portanto, geram os mesmos valores singulares. Quando $\boldsymbol{A}$ não é quadrada, $\boldsymbol{A}\boldsymbol{A}^T$ possui dimensão $m \times m$ e $\boldsymbol{A}^T\boldsymbol{A}$ possui dimensão $n \times n$. Assim, uma delas pode apresentar mais autovalores que a outra. Entretanto, os autovalores excedentes são iguais a zero, de modo que o número de autovalores não nulos é limitado por $\min(m,n)$.


## Exemplo

Veja que a matriz $\boldsymbol{A}$ mostrada abaixo não é quadrada e, obviamente, não simétrica:

\begin{equation}
\boldsymbol{A} = \begin{bmatrix}4 & 1 & 3 &2 \\ 8 & 7 & -2 & 5 \\ 1 & 5 & 4 & 3 \end{bmatrix}
\end{equation}

Aplicaremos SVD nela.

In [ ]:
A = np.array([[4, 1, 3, 2],
              [8, 7, -2, 5],
              [1, 5, 4, 3]])

In [ ]:
U, S, VT = np.linalg.svd(A, full_matrices=False)
print('A saída do SVD são 3 matrizes\n Veja que a matriz V é transposta\n')
print(' Matriz Sigma:')
print(S)
print('\n Matriz U:')
print(U)
print('\n Matriz V transposta:')
print(VT)

Veja que usamos `full_matrices=False`, quando calculamos SVD.

Com `full_matrices=True`, o `NumPy` devolve uma base completa do espaço de saída e de entrada, incluindo o autovalor com autovetor nulo da matriz $\boldsymbol V$. COmo discutimos anteriormente, $U$ terá 3 e $V$ terá 4 autovetores possíveis. Mas apenas 3 valores singulares existem, porque $\min(3,4)=3$.

Então sobra uma coluna/direção em V associada a valor singular zero. É por isso que, se você `full_matrices=True` (default) você precisa da matriz $\boldsymbol{\Sigma}$ retangular $3 \times 4$,
```
Sigma = np.zeros((3,4))
Sigma[:3,:3] = np.diag(S)
```
Com `full_matrices=False`, o `NumPy` remove automaticamente essas direções extras que não contribuem para a reconstrução. Ele devolve apenas as 'k=min(m,n)' direções relevantes.

Veja que recuperamos a matriz $\boldsymbol{A}$ fazendo a multiplicação:
$$
\boldsymbol{A}=\boldsymbol{U}\boldsymbol{\Sigma}\boldsymbol{V}^T
$$

In [ ]:
A = np.linalg.multi_dot([U, np.diag(S), VT])
print('Matriz A recuperada:')
print(A)

Uma vez obtida a decomposição em valores singulares de uma matriz $\boldsymbol{A}$,

$$
\boldsymbol{A}=\boldsymbol{U}\boldsymbol{\Sigma}\boldsymbol{V}^T,
$$

é possível interpretar essa decomposição como uma soma de matrizes de posto 1. Se $\sigma_1,\sigma_2,\ldots,\sigma_r$ são os valores singulares não nulos de $\boldsymbol{A}$, $u_1,u_2,\ldots,u_r$ são as colunas de $\boldsymbol{U}$ e $v_1,v_2,\ldots,v_r$ são as colunas de $\boldsymbol{V}$, então:

$$
\boldsymbol{A} =
\sigma_1 u_1 v_1^T
+
\sigma_2 u_2 v_2^T
+\cdots+
\sigma_r u_r v_r^T.
$$

Cada termo da soma,

$$
\sigma_i u_i v_i^T,
$$

é uma matriz de posto 1 obtida pelo produto externo entre os vetores $u_i$ e $v_i$, multiplicado pelo valor singular correspondente $\sigma_i$.

Essa interpretação é extremamente importante, pois mostra que a matriz original pode ser construída pela soma de componentes independentes. Além disso, como os valores singulares são ordenados de forma decrescente,

$$
\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r,
$$

os primeiros termos da soma costumam concentrar a maior parte da informação presente nos dados.

Assim, uma aproximação de posto $k$ da matriz pode ser obtida utilizando apenas os $k$ maiores valores singulares:

$$
\boldsymbol{A}_k
=\sum_{i=1}^{k}
\sigma_i u_i v_i^T.
$$

Quando $k<r$, obtém-se uma representação mais compacta da matriz original, com perda controlada de informação. Essa propriedade é amplamente utilizada em compressão de imagens, redução de dimensionalidade e no cálculo das chamadas eigenfaces para reconhecimento de rostos.


Voltando ao nosso exemplo anterior, a decomposição da matriz em somas de matrizes de posto 1 é dada por

A saída do SVD são três matrizes. Observe que a terceira matriz retornada pelo algoritmo é $\boldsymbol{V}^T$, e não $\boldsymbol{V}$.

$$
\boldsymbol{\Sigma}
=
\begin{bmatrix}
13.4490 & 0 & 0\\
0 & 5.6123 & 0\\
0 & 0 & 3.2599
\end{bmatrix}
$$

$$
\boldsymbol{u}_1=
\begin{bmatrix}
0.3166\\
0.8613\\
0.3974
\end{bmatrix},
\qquad
\boldsymbol{u}_2=
\begin{bmatrix}
0.3326\\
-0.4932\\
0.8038
\end{bmatrix},
\qquad
\boldsymbol{u}_3=
\begin{bmatrix}
-0.8883\\
0.1223\\
0.4426
\end{bmatrix}.
$$

As linhas de $\boldsymbol{V}^T$ correspondem aos vetores $\boldsymbol{v}_i^T$:

$$
\boldsymbol{v}_1^T=
\begin{bmatrix}
0.6360 & 0.6196 & 0.0607 & 0.4559
\end{bmatrix},
$$

$$
\boldsymbol{v}_2^T=
\begin{bmatrix}
-0.3227 & 0.1603 & 0.9265 & 0.1089
\end{bmatrix},
$$

$$
\boldsymbol{v}_3^T=
\begin{bmatrix}
-0.6541 & 0.6690 & -0.3494 & 0.0499
\end{bmatrix}.
$$

Portanto,

\begin{align}
\boldsymbol{A}
=\\
& 13.4490
\begin{bmatrix}
0.3166\\
0.8613\\
0.3974
\end{bmatrix}
\begin{bmatrix}
0.6360 & 0.6196 & 0.0607 & 0.4559
\end{bmatrix}
+\\
& 5.6123
\begin{bmatrix}
0.3326\\
-0.4932\\
0.8038
\end{bmatrix}
\begin{bmatrix}
-0.3227 & 0.1603 & 0.9265 & 0.1089
\end{bmatrix}
+\\
& 3.2599
\begin{bmatrix}
-0.8883\\
0.1223\\
0.4426
\end{bmatrix}
\begin{bmatrix}
-0.6541 & 0.6690 & -0.3494 & 0.0499
\end{bmatrix}.
\end{align}


Essa é exatamente a decomposição:

\begin{align}
\boldsymbol{A}
=
\sigma_1 \boldsymbol{u}_1 \boldsymbol{v}_1^T
+
\sigma_2 \boldsymbol{u}_2 \boldsymbol{v}_2^T
+
\sigma_3 \boldsymbol{u}_3 \boldsymbol{v}_3^T.
\end{align}

In [ ]:
A_rec = np.zeros(A.shape)

for k in range(len(S)):
    A_rec += S[k] * np.outer(U[:, k], VT[k, :])

    print(f'Aproximação usando os {k+1} maiores valores singulares:')
    print(A_rec)
    print('-'*50)

# Aplicação em Eigenfaces

Em vez de olhar para uma fotografia como milhares de pixels independentes, a técnica de Eigenfaces (que podemos traduzir como "rostos essenciais" ou "rostos característicos") parte de uma ideia simples: será que é possível descrever qualquer rosto a partir de um pequeno conjunto de padrões básicos?

As *Eigenfaces* são exatamente esses padrões. Elas correspondem a um conjunto de imagens que capturam as principais variações presentes em uma grande coleção de rostos, como mudanças de iluminação, formato da face, posição dos olhos e distribuição de sombras. A proposta é representar o rosto de qualquer pessoa como uma combinação linear dessas imagens básicas.

Em outras palavras, em vez de armazenar milhares de valores de pixels, podemos descrever um rosto por meio de alguns coeficientes. Por exemplo, o rosto do João pode ser representado como uma combinação de $40%$ da Eigenface 1, somado a $15%$ da Eigenface 2, subtraindo $5%$ da Eigenface 3, e assim por diante. Esses coeficientes funcionam como uma representação compacta da imagem original.

Para ilustrar essa ideia, utilizaremos um conjunto de imagens de rostos disponível na biblioteca `sklearn.datasets`.


In [ ]:
from sklearn.datasets import fetch_olivetti_faces
data = fetch_olivetti_faces()
imgs = data.images

Veja que o dataset possui $400$ imagens (`nimg`) de dimensão $64 \times 64$ (`width, height`)

In [ ]:
print(imgs.shape)
nimg,width,height=imgs.shape

Veja algumas imagens da biblioteca...

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 8))
plt.subplots_adjust(wspace=0.4)

for i in range(0, 5):
    axes[i].imshow(imgs[i*20], cmap='gray')

plt.show()

### Exemplo: Seleção de uma face

Transforma-se os dados de cada imagem em uma matriz. O tensor `imgs` é redimensionado para $4096 \times 400$ e armazenado na matriz `M` - cada linha possui $4096 (64\times 64)$ pixels da mesma face. Pode-se pensar também que cada coluna é o mesmo pixel de cada face diferente.

<center><img src='https://drive.google.com/uc?export=view&id=1DXACJKfODwyiJw92eWSenN8TswrVwGDP' width="400"></center>

Obviamente, para recuperar uma face `facenumber` em particular, basta:
```
y = M[facenumber, :]
```
Você pode escolher qualquer valor para `facenumber` entre 0 e 399.

Antes de plotar a imagem, redimensionamos vetor `y` para uma matriz de $64\times 64$.

In [ ]:
# M tem formato (nimg=400, width*height=64*64=4096)
# Cada linha é uma imagem
M = imgs.reshape(nimg, width*height)

# Escolha a face
facenumber = 40
y = M[facenumber].reshape(height, width)

print("Formatos:", M.shape, y.shape)
plt.imshow(y, cmap='gray')
plt.show()

## O Rosto Médio e a Centralização dos Dados

No algoritmo de *Eigenfaces*, o primeiro passo fundamental é calcular o *Rosto Médio* (*Mean Face*) do nosso conjunto de dados e centralizar as imagens.

### 1. O que o Rosto Médio representa?
O rosto médio é a média matemática de todos os pixels, na mesma posição, de todas as imagens do dataset. Visualmente, ele se parece com um rosto esfumaçado, meio andrógino e genérico. Ele representa as características básicas em comum que todos os rostos da sua base de dados compartilham (como a presença de dois olhos, nariz, boca e a iluminação média).

### 2. Por que precisamos centralizar os dados (Subtrair a média)?
O objetivo do SVD aplicado ao reconhecimento facial é capturar as *variações* e as *diferenças* entre as imagens, e não o que elas têm em comum.

### 3. Como o Rosto Médio será usado futuramente?
* **Para extrair as Eigenfaces:** A matriz centralizada será usada para calcular a matriz de covariância, e é a partir dela que extrairemos os autovetores (as *Eigenfaces*).
* **No Reconhecimento Facial:** Quando o sistema receber uma imagem de um rosto novo para reconhecer, a primeira coisa que o programa deverá fazer é subtrair o Rosto Médio dessa nova imagem antes de analisá-la.
* **Na Reconstrução de Imagens:** Para reconstruir e visualizar um rosto a partir dos seus componentes principais compactados, fazemos o processo inverso: calculamos os pesos matemáticos e, no último passo, **somamos o Rosto Médio de volta** para que a imagem recupere a sua aparência real.

In [ ]:
# Cálculo do Rosto Médio
# Como as imagens são linhas na matriz M, tiramos a média ao longo do eixo das linhas (axis=0)
mean_face = np.mean(M, axis=0, keepdims=True)

# Subtração da média de M para Centralizar os dados
M_centered = M - mean_face

print(f"Dimensão da Matriz M original: {M.shape}")
print(f"Dimensão do Rosto Médio: {mean_face.shape}")
print(f"Dimensão da Matriz M Centralizada: {M_centered.shape}")

# Visualizar o Rosto Médio
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)

plt.imshow(mean_face.reshape((height, width)), cmap='gray')
plt.title("Rosto Médio")
plt.axis('off')

# Visualizar uma face específica (ex: índice 40) centralizada (M_centered)
facenumber = 40
y_centered = M_centered[facenumber, :]

plt.subplot(1, 2, 2)
plt.imshow(y_centered.reshape((height, width)), cmap='gray')
plt.title(f"Face {facenumber} Centralizada")
plt.axis('off')

plt.tight_layout()
plt.show()

Veja que, ao remodelar o vetor `y_centered`, ve-se a face original "escura", pois os valores de pixel agora são as variações em torno da média. Isso ocorre porque a imagem centralizada não mostra mais um *rosto completo*, ela mostra o quanto aquele rosto se desvia do padrão. Se o desvio for pequeno, o pixel fica cinza escuro ou preto.

Como a maioria dos rostos humanos compartilha a mesma estrutura básica (olhos, nariz e boca nos mesmos lugares), o *rosto médio* já carrega quase toda a iluminação e formas comuns. Quando você subtrai essa média, você remove a iluminação geral. O que sobra na matriz `M_centered` são apenas os detalhes únicos daquela face específica (uma sobrancelha mais grossa, um nariz um pouco mais largo, uma sombra diferente). Como a maior parte das características comuns virou zero ou valores negativos, a imagem resultante perde o fundo claro e o brilho normal, parecendo uma *versão escura*, um negativo ou um *fantasma* onde só os detalhes fora do padrão aparecem.



## Extraindo as Eigenfaces com SVD

Agora que nossos dados estão centralizados, podemos aplicar a **Decomposição em Valores Singulares (SVD)**. Ao decompormos nossa matriz $M_{centered}$, o SVD nos devolve três matrizes:

$$
\boldsymbol{M}_{centered}
=
\boldsymbol{U}\boldsymbol{\Sigma}\boldsymbol{V}^T
$$
ou
$$
\boldsymbol{M}_{centered}
=
\sigma_1 (\mathbf{u}_1 \mathbf{v}_1^T) + \sigma_2 (\mathbf{u}_2 \mathbf{v}_2^T) + \sigma_3 (\mathbf{u}_3 \mathbf{v}_3^T) + \dots$$

Onde:
* **As Eigenfaces ($\boldsymbol{V}^T$):** Cada linha de $V^T$ contém as direções de máxima variação dos pixels. No contexto de imagens, cada linha tem o mesmo tamanho de uma imagem vetorizada (4096 pixels). Quando reorganizamos essa linha de volta para o formato bidimensional, temos as famosas **Eigenfaces** (ou *rostos fantasmas*), como você poderá perceber nas imagens que obterá rodando o código a seguir. Ou seja, cada linha de $V^T$ representa um "rosto base" que captura as características morfológicas primárias do conjunto de dados.
* **A Importância delas ($\boldsymbol{\Sigma}$):** A matriz diagonal $\Sigma$ (representada pelo vetor `S` no código) contém os **Valores Singulares**. Eles nos dizem a importância matemática (a "energia" ou magnitude) que cada Eigenface tem para explicar as diferenças entre os rostos. As primeiras linhas de $V^T$ têm valores singulares gigantescos, enquanto as últimas têm valores muito pequenos.
* **Os Coeficientes de Projeção ($\boldsymbol{U}$ e $\boldsymbol{\Sigma}$ juntos):** A A matriz $\boldsymbol{U}$ está diretamente relacionada às pessoas que compõem o nosso dataset. Nela, cada linha é a *assinatura digital única* de uma pessoa específica (mostrando a proporção de feições que ela possui) e cada coluna mostra como uma característica estrutural isolada varia ao longo de todos os indivíduos do grupo. No entanto, $\boldsymbol{U}$ indica apenas direções puras; ela precisa de uma escala. A matriz $\boldsymbol{\Sigma}$ dá a intensidade ou o peso real para cada uma dessas características. Na prática, para reconstruirmos a foto original de uma pessoa do dataset (por exemplo, a foto 160), fazemos uma combinação linear das Eigenfaces ($\boldsymbol{V}^T$), onde os pesos vêm da linha 160 da matriz $U$ multiplicada pelos valores de $\Sigma$.

<center><img src='https://drive.google.com/uc?export=view&id=1khxhPSkRYBT9WrvZ7DFvHgqm65bu7_Y4' width="600"></center>

In [ ]:
mean_face = np.mean(M, axis=0, keepdims=True)
M_centered = M - mean_face

#SVD
U, S, VT = np.linalg.svd(M_centered, full_matrices=False)

# Visualização das Eigenfaces
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
plt.subplots_adjust(wspace=0.3, hspace=0.3)

eigens=[0,1,2,3,99,199,299,399]
for i,j in zip(range(0,8),eigens):
    eigenface = VT[j, :].reshape((height, width))
    axes[i // 4, i % 4].imshow(eigenface, cmap='jet')
    axes[i // 4, i % 4].set_title(f"$v_{{{j+1}}}$", fontsize=14)
    axes[i // 4, i % 4].axis('off')

plt.suptitle("Eigenfaces (Vetores Singulares de V^T)", fontsize=16, y=0.98)
plt.show()

Como subtraimos o rosto médio corretamente, a matriz $\boldsymbol{M}_{centered}$ contém apenas o que se afasta da média. O primeiro vetor singular $\boldsymbol{v}_1$(a primeira Eigenface) tem a missão matemática de apontar para a maior direção de variação de todo o dataset. O vetor $\boldsymbol{v}_1$ mostra claramente o contorno de óculos proeminentes e uma linha marcante de barba/bigode ou boca. $v_1^T$ a $v_4^T$ mostram estruturas claras (rosto, óculos, iluminação), ainda levemente visível em $v_{100}^T$. Em $v_{200}^T$  $v_{300}^T$ o rosto começa a sumir, virando um *fantasminha* cheio de texturas ásperas. Ao final, $v_{400}^T$ é puro ruído de alta frequência (chuvisco).

Agora, vamos recuperar a imagem da pessoa número 160 da lista:

In [ ]:
facenumber = 160
face160 = (U[facenumber, :] @ np.diag(S) @ VT) + mean_face
plt.imshow(face160.reshape((width,height)), cmap='gray');

## Quantas componentes precisamos para definir um rosto?
Um rosto humano possui uma quantidade massiva de pixels, mas muita dessa informação é redundante. Com o SVD, as primeiras Eigenfaces capturam as maiores estruturas de variação (como o formato do queixo, iluminação lateral, se a pessoa usa óculos ou tem barba). À medida que avançamos para as últimas Eigenfaces, elas passam a capturar apenas detalhes muito sutis ou ruídos.

Olhando para o gráfico de **Variância Acumulada**, conseguimos decidir visualmente o *trade-off* perfeito: quantas componentes precisamos manter para reter, por exemplo, 90% ou 95% da informação essencial de um rosto, reduzindo drasticamente o tamanho do problema antes de alimentarmos qualquer classificador no futuro.

### Variância Explicada no SVD

Quando aplicamos o SVD na nossa matriz de dados centralizados $\boldsymbol{M}_{centered}$, estamos decompondo a informação original em várias "camadas" independentes (as Eigenfaces). A **Variância Explicada** é a métrica que nos diz quanta informação geométrica da nossa base de dados cada uma dessas Eigenfaces conseguiu capturar.

Em termos simples, a variância total mede o *espalhamento* ou a quantidade de detalhes e diferenças que existem entre todos os rostos do dataset. Dizer que a primeira Eigenface explica 40% da variância significa que, sozinha, ela reconstrói quase metade de todas as diferenças visuais que diferenciam os rostos das pessoas do dataset.

#### Por que a Variância é dada por $\sigma^2$?

> Essa parte do Notebook é uma demonstração matemática totalmente optativa. Caso queira pular este subitem, não irá prejudicar seu entendimento do todo.

A variância total de uma matriz de dados centralizada $\boldsymbol{M}_{centered}$ é equivalente à soma de todos os seus elementos elevados ao quadrado (o que na álgebra linear chamamos de **Norma de Frobenius** ao quadrado, $\|\boldsymbol{M}_{centered}\|_F^2$).

Geometricamente, a variância/energia total armazenada na nossa matriz pode ser calculada através do traço ($\text{Tr}$) do produto da matriz por sua transposta:

$$\text{Variância Total} \propto \|\boldsymbol{M}_{centered}\|_F^2 = \text{Tr}(\boldsymbol{M}_{centered}^T \boldsymbol{M}_{centered})$$

Substituindo a decomposição SVD ($\boldsymbol{M}_{centered} = \boldsymbol{U}\boldsymbol{\Sigma}\boldsymbol{V}^T$) dentro do produto:

$$\boldsymbol{M}_{centered}^T \boldsymbol{M}_{centered} = (\boldsymbol{U}\boldsymbol{\Sigma}\boldsymbol{V}^T)^T (\boldsymbol{U}\boldsymbol{\Sigma}\boldsymbol{V}^T)$$

Aplicando a propriedade da transposta do produto $((ABC)^T = C^TB^TA^T)$:

$$\boldsymbol{M}_{centered}^T \boldsymbol{M}_{centered} = \boldsymbol{V} \boldsymbol{\Sigma}^T \boldsymbol{U}^T \boldsymbol{U} \boldsymbol{\Sigma} \boldsymbol{V}^T$$

Como $\boldsymbol{U}$ é uma matriz ortogonal por definição do SVD, suas colunas são vetores ortonormais, o que significa que $\boldsymbol{U}^T \boldsymbol{U} = \boldsymbol{I}$ (Matriz Identidade). A equação se simplifica para:

$$\boldsymbol{M}_{centered}^T \boldsymbol{M}_{centered} = \boldsymbol{V} \boldsymbol{\Sigma}^2 \boldsymbol{V}^T$$

Como $\boldsymbol{V}$ também é uma matriz ortogonal, multiplicá-la por $\boldsymbol{\Sigma}^2$ representa apenas uma rotação rígida no espaço coordenado, o que **não altera o comprimento ou a energia total dos vetores**.

Portanto, a Norma de Frobenius (a variância total) de $\boldsymbol{M}_{centered}$ é conservada e se reduz puramente à soma dos elementos da matriz diagonal $\boldsymbol{\Sigma}^2$:

$$\|\boldsymbol{M}_{centered}\|_F^2 = \|\boldsymbol{\Sigma}\|_F^2 = \sigma_1^2 + \sigma_2^2 + \sigma_3^2 + \dots + \sigma_k^2$$

### Conclusão para nosso código:
Como as Eigenfaces ($\boldsymbol{V}^T$) representam direções completamente perpendiculares (ortogonais) entre si, o SVD consegue fatiar a variância total do dataset perfeitamente. Cada valor singular elevado ao quadrado ($\sigma_i^2$) quantifica exatamente a **porcentagem absoluta de informação** que aquela respectiva Eigenface carrega.

É por isso que, para descobrir a relevância relativa de cada componente no Python, fazemos:
```python
variancia_explicada = (S**2) / np.sum(S**2)

In [ ]:
variancia_explicada = (S**2) / np.sum(S**2)
variancia_acumulada = np.cumsum(variancia_explicada) # calcula a soma acumulada

plt.figure(figsize=(10, 4))
plt.plot(variancia_acumulada, marker='o', color='b', linestyle='--')
plt.axhline(y=0.90, color='r', linestyle=':', label='90% de Informação Retida')
plt.axhline(y=0.95, color='g', linestyle=':', label='95% de Informação Retida')

plt.title("Variância Acumulada pelas Eigenfaces")
plt.xlabel("Número de Componentes (Eigenfaces)")
plt.ylabel("Porcentagem de Informação Explicada")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

componentes_90 = np.argmax(variancia_acumulada >= 0.90) + 1
print(f"\n-> Para reter 90% da informação dos rostos, precisamos de apenas {componentes_90} componentes!")

### Reconstrução

Veja que, com a projeção do vetor `x` particularizado para face `facenumber=160` nas direções principais conseguimos recuperar as características particulares de forma satisfatória usando os primeiros 80 valores singulares.

In [ ]:
x = np.zeros((nimg, 1))
facenumber = 160
x[facenumber, 0] = 1

fig, axes = plt.subplots(2, 4, figsize=(14, 8))
fig.suptitle("Reconstrução da imagem usando os primeiros k valores singulares", fontsize=16)
plt.subplots_adjust(wspace=0.1, hspace=0.1)

axes[0, 0].imshow(imgs[facenumber], cmap='gray')
axes[0, 0].set_title("Imagem original", fontsize=14)
axes[0,0].axis('off')

k_list = [1, 6, 10, 15, 20, 35, componentes_90]
for i in range(1, 8):
    # Reconstrução da matriz usando os primeiros k valores singulares
    k = k_list[i-1]
    mat_approx = (U[facenumber, :k] @ np.diag(S[:k]) @ VT[:k, :]) + mean_face

    axes[i // 4, i % 4].imshow(mat_approx.reshape((width,height)), cmap='gray')
    axes[i // 4, i % 4].set_title("k = {}".format(k), fontsize=14)
    axes[i // 4, i % 4].axis('off')

plt.show()

# Para que tudo isso isso irá servir?

As primeiras componentes obtidas pelo SVD capturam os padrões mais importantes presentes nas imagens, como formato do rosto, posição dos olhos, nariz e boca. Já as componentes menos relevantes tendem a representar detalhes muito específicos, variações de iluminação e ruídos (a estática da câmera).

Nas próximas duas aulas, portanto, vocês irão construir um programa para reconhecimento facial de qualquer colega seu do curso de IA. Para isso, serão 3 etapas:
1. Geração de Eigenfaces - utilização do banco de dados *Labeled Faces in the Wild* (LFW) em conjunto com as fotos de seus colegas, para extrair as características principais de uma face humana. Em outras palavras, antes de ensinar o SVM a distinguir pessoas, precisamos encontrar uma representação compacta dos rostos que preserve as características mais importantes e descarte informações pouco úteis.
2. Criação um classificador para reconhecimento facial de seus colegas - projetaremos os rostos dos colegas nas componentes mais relevantes e utilizaremos essas informações como entrada para um classificador SVM, capaz de realizar reconhecimento facial;
3. Utilização do classificador em tempo real na competição em sala.

__Não perca a próxima aula, será muito boa!__

<center><img src='https://drive.google.com/uc?export=view&id=1LYiTAE2KG5dJf_qIoVKOluUhzrK-AmfP' width="600"></center>

